---
numbering: false
---

# 4.1. The Gradient Vector

The big theme in our course so far has been the three-step modeling recipe, of choosing a model, choosing a loss function, and then minimizing empirical risk (i.e. average loss) to find optimal model parameters.

In [Chapter 1.4](../supervised-learning/simple-linear-regression), we used calculus to find the slope $w_1^*$ and intercept $w_0^*$ that minimized mean squared error,

$$R_\text{sq}(w_0, w_1) = \frac{1}{n} \sum_{i=1}^n (y_i - (w_0 + w_1 x_i))^2$$ 

by taking partial derivatives with respect to $w_0$ and $w_1$ and setting them to zero.

Then, in Chapters 2 and 3, we focused on the multiple linear regression model and the squared loss function, which saw us minimize

$$R_\text{sq}(\vec w) = \frac{1}{n} \lVert \vec y - X \vec w \rVert^2 = \frac{1}{n} \sum_{i=1}^n (y_i - \vec w \cdot \text{Aug}(\vec x_i))^2$$

where $X$ is the $n \times (d + 1)$ design matrix, $\vec y \in \mathbb{R}^n$ is the observation vector, and $\vec w \in \mathbb{R}^{d+1}$ is the parameter vector we're trying to pick. In [Chapter 2.10](../vectors-and-matrices/projection-2), we minimized $R_\text{sq}(\vec w)$ by arguing that the optimal $\vec w^*$ had to create an error vector, $\vec e = \vec y - X \vec w^*$, that was orthogonal to $\text{colsp}(X)$, which led us to the normal equations.

It turns out that there's a way to use our calculus-based approach from Chapter 1.4 to minimize the more general version of $R_\text{sq}$ for any $d$, that **doesn't** involve computing $d$ partial derivatives. To see how this works, we need to define a new object, the **gradient vector**, which we'll do here in Chapter 4.1. After we're familiar with how the gradient vector works, we'll use it to build a new approach to function minimization, one that works even when there isn't a closed-form solution for the optimal parameters: that technique is called **gradient descent**, which we'll see in Chapter 4.2.

---

## Domain and Codomain

As we saw in [Chapter 2.9](../vectors-and-matrices/inverses) when we first introduced the concept of the inverse of a matrix, the notation

$$f: \mathbb{R}^d \to \mathbb{R}^n$$

means that $f$ is a function whose inputs are vectors with $d$ components and whose outputs are vectors with $n$ components. $\mathbb{R}^d$ is the **domain** of the function, and $\mathbb{R}^n$ is the **codomain**. I've used $d$ and $n$ to match the notation we've used for matrices and linear transformations. In general, if $A$ is an $n \times d$ matrix, then any vector $\vec x$ multiplied by $A$ (on the right) must be in $\mathbb{R}^d$ and the result $A \vec x$ will be in $\mathbb{R}^n$.

Given this framing, consider the following four types of functions.

| Type | Domain and Codomain | Examples |
|------|----------------------|----------|
| **Scalar-to-scalar** | $f: \mathbb{R} \to \mathbb{R}$ | $f(x) = x^2 + \sin(x)$<br>$R_\text{sq}(w) = \frac{1}{n} \sum_{i=1}^n (y_i - w)^2$ |
| **Vector-to-scalar** | $f: \mathbb{R}^d \to \mathbb{R}$ | $f(\vec x) = \vec x^T \vec x$<br>$f(\vec x) = (x_1 - 1)^2 + (x_1 - x_2)^2 + 3$<br>$R_\text{sq}(\vec w) = \frac{1}{n} \lVert \vec y - X \vec w \rVert^2$ |
| **Scalar-to-vector** | $f: \mathbb{R} \to \mathbb{R}^n$ | $f(x) = \underbrace{\begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} + x \begin{bmatrix} 3 \\ 0 \\ -4 \end{bmatrix}}_{\text{parametric form of a line}}$<br><br>$f(x) = \begin{bmatrix} x^2 + e^{x} \\ -3 \end{bmatrix}$ |
| **Vector-to-vector** | $f: \mathbb{R}^d \to \mathbb{R}^n$ | $\underbrace{f(\vec x) = \begin{bmatrix} 3 & 4 \\ 2 & 0 \\ 0 & 1 \end{bmatrix} \vec x}_{\text{linear transformation}}$ <br><br> $f(\vec x) = \begin{bmatrix} x_1^2 + x_1x_2 + \cos(x_1^4) \\ 3x_1x_2 + 4 \\ 5x_1 \\ x_2 / x_3 \end{bmatrix}$ |

The first two types of functions are "scalar-valued", while the latter two are "vector-valued". These are not the only types of functions that exist; for instance, the function $f(A) = \text{rank}(A)$ is a matrix-to-scalar function.

The type of function we're most concerned with at the moment are **vector-to-scalar** functions, i.e. functions who take in a single vector and output a single number.

$$R_\text{sq}(\vec w) = \frac{1}{n} \lVert \vec y - X \vec w \rVert^2$$

is one such function.

---

## Rates of Change

Let's think from the perspectives of **rates of change**, since ultimately what we're building towards is a technique for minimizing functions. We're most familiar with the concept of rates of change for scalar-to-scalar functions; the **derivative** of a scalar-to-scalar function describes its instantaneous rate of change at any point in its domain.

$$f(x) = x^2 + \sin(x) \implies \frac{\text{d}f}{\text{d}x} = 2x + \cos(x)$$

---

## The Gradient Vector

:::{note} Definition: Gradient Vector
Suppose $f: \mathbb{R}^d \to \mathbb{R}$ is a vector-to-scalar function. The **gradient vector** of $f$, denoted $\nabla f(\vec x)$, is the vector of partial derivatives of $f$:

$$\nabla f(\vec x) = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_d} \end{bmatrix}$$

The gradient itself is a vector-to-vector function; it takes in a vector $\vec x \in \mathbb{R}^d$ and outputs a new vector in $\mathbb{R}^d$, describing the rates of change of $f$ along each dimension. The gradient at a vector describes the **direction of steepest ascent** of $f$ at that point, i.e. the direction in which $f$ increases the most quickly.
:::

As usual, we'll start with an example. Suppose $x \in \mathbb{R}^2$, and let

$$f(\vec x) = x_1 e^{-x_1^2 - x_2^2}$$

We'll need to graph $f$ in 3 dimensional space. One way to do so is to draw $f$ as a surface in 3D.

In [63]:
def make_3D_surface(f, lim, xaxis_title, yaxis_title, zaxis_title, title):
    x1_range = np.linspace(-lim, lim, 100)
    x2_range = np.linspace(-lim, lim, 100)
    x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)
    z_values = f(x1_grid, x2_grid)

    fig = go.Figure(data=[
        go.Surface(
            x=x1_grid,
            y=x2_grid,
            z=z_values,
            colorscale='RdBu_r',
            showscale=False,
        )
    ])
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(
                title=xaxis_title, gridcolor="#f0f0f0",
                showbackground=True, showline=True, linecolor="black", linewidth=1,
                tickfont=dict(family='Palatino', size=10),
                backgroundcolor="white"
            ),
            yaxis=dict(
                title=yaxis_title, gridcolor="#f0f0f0",
                showbackground=True, showline=True, linecolor="black", linewidth=1,
                tickfont=dict(family='Palatino', size=10),
                backgroundcolor="white"
            ),
            zaxis=dict(
                title=zaxis_title, gridcolor="#f0f0f0",
                showbackground=True, showline=True, linecolor="black", linewidth=1,
                tickfont=dict(family='Palatino', size=10),
                backgroundcolor="white"
            ),
            aspectratio=dict(x=1, y=1, z=1),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1))
        ),
        width=1000,
        height=800,
        margin=dict(l=65, r=50, b=65, t=90),
        font=dict(family='Palatino', size=16, color="#222"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        showlegend=False,
    )
    return fig


def make_3D_contour(
    f,
    dfx1=None,
    dfx2=None,
    lim=2,
    xaxis_title="w1",
    yaxis_title="w2",
    title=None,
    with_gradient=False,
    grad_point=None,
    neg=False,
    contour_kwargs=None,
):
    import numpy as np
    import plotly.graph_objects as go

    x1_range = np.linspace(-lim, lim, 100)
    x2_range = np.linspace(-lim, lim, 100)
    x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)
    z_values = f(x1_grid, x2_grid)
    if contour_kwargs is None:
        contour_kwargs = {}

    base_contour = dict(
        z=z_values,
        x=x1_range,
        y=x2_range,
        colorscale='RdBu_r',
        showscale=False,
        contours=dict(
            showlabels=True,
            labelfont=dict(size=12, color='white', family='Palatino'),
        ),
    )
    base_contour.update(contour_kwargs)
    fig = go.Figure(data=[
        go.Contour(**base_contour)
    ])
    if with_gradient and dfx1 is not None and dfx2 is not None and grad_point is not None:
        w1_start, w2_start = grad_point
        a = 0.1
        grad_x, grad_y = dfx1(w1_start, w2_start), dfx2(w1_start, w2_start)
        if neg:
            w1_end, w2_end = w1_start - grad_x * a, w2_start - grad_y * a
        else:
            w1_end, w2_end = w1_start + grad_x * a, w2_start + grad_y * a

        fig.add_trace(go.Scatter(
            x=[w1_start, w1_end], y=[w2_start, w2_end],
            mode='lines+markers',
            line=dict(color='red' if neg else 'gold', width=5),
            showlegend=False,
        ))

        alph = 0.01
        if neg:
            ax = w1_start - grad_x * alph
            ay = w2_start - grad_y * alph
        else:
            ax = w1_start + grad_x * alph
            ay = w2_start + grad_y * alph

        fig.add_annotation(
            x=w1_end, y=w2_end,
            ax=ax, ay=ay,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=3, arrowsize=1.5,
            arrowwidth=3, arrowcolor='red' if neg else 'gold'
        )
    if with_gradient and grad_point is not None:
        col = 'red' if neg else 'gold'
        text = 'Negative of the Gradient Vector' if neg else 'Gradient Vector'
        final_title = f'<span style="color:{col}"><b>{text}</b></span> at Point ({grad_point[0]}, {grad_point[1]})'
    else:
        final_title = title or ''
    fig.update_layout(
        title=final_title,
        xaxis=dict(
            title=xaxis_title, gridcolor="#f0f0f0",
            showline=True, linecolor="black", linewidth=1,
            tickfont=dict(family='Palatino', size=10)
        ),
        yaxis=dict(
            title=yaxis_title, gridcolor="#f0f0f0",
            showline=True, linecolor="black", linewidth=1,
            tickfont=dict(family='Palatino', size=10)
        ),
        width=800,
        height=700,
        margin=dict(l=65, r=50, b=65, t=90),
        showlegend=False,
        font=dict(family='Palatino', size=16, color="#222"),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    return fig


def show_surface_and_contour_side_by_side(
    f,
    dfx1=None,
    dfx2=None,
    lim=2,
    xaxis_title="x₁",
    yaxis_title="x₂",
    zaxis_title="f(x₁, x₂)",
    surface_title="3D Surface",
    contour_title="Contour",
    with_gradient=False,
    grad_point=None,
    neg=False,
    contour_kwargs=None,
    width=800,
    height=1200,
):
    """
    Display both 3D surface and 2D contour stacked vertically (2 rows, 1 column) with shared stylings.
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    surface_fig = make_3D_surface(
        f=f,
        lim=lim,
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        zaxis_title=zaxis_title,
        title=surface_title,
    )
    contour_fig = make_3D_contour(
        f=f,
        dfx1=dfx1, dfx2=dfx2, lim=lim,
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        title=contour_title,
        with_gradient=with_gradient,
        grad_point=grad_point,
        neg=neg,
        contour_kwargs=contour_kwargs,
    )
    subplot_fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(surface_title, contour_title),
        specs=[[{'type': 'surface'}], [{'type': 'xy'}]],
        vertical_spacing=0.13
    )
    # Transfer surface traces (3D plot)
    for trace in surface_fig.data:
        subplot_fig.add_trace(trace, row=1, col=1)
    for trace in contour_fig.data:
        subplot_fig.add_trace(trace, row=2, col=1)

    # Transfer contour annotations (gradient arrow)
    if hasattr(contour_fig, "layout") and "annotations" in contour_fig.layout:
        subplot_fig.layout.annotations += tuple(contour_fig.layout.annotations)

    # Layout: shared formatting
    subplot_fig.update_layout(
        height=height,
        width=width,
        margin=dict(l=40, r=40, b=60, t=100),
        font=dict(family='Palatino', size=16, color="#222"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        showlegend=False,
    )

    subplot_fig.update_scenes(
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        zaxis_title=zaxis_title,
        xaxis=dict(
            gridcolor="#f0f0f0", showbackground=True,
            showline=True, linecolor="black", linewidth=1,
            tickfont=dict(family='Palatino', size=10),
            backgroundcolor="white"
        ),
        yaxis=dict(
            gridcolor="#f0f0f0", showbackground=True,
            showline=True, linecolor="black", linewidth=1,
            tickfont=dict(family='Palatino', size=10),
            backgroundcolor="white"
        ),
        zaxis=dict(
            gridcolor="#f0f0f0", showbackground=True,
            showline=True, linecolor="black", linewidth=1,
            tickfont=dict(family='Palatino', size=10),
            backgroundcolor="white"
        ),
        aspectratio=dict(x=1, y=1, z=1),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1))
    )
    subplot_fig.update_xaxes(
        title_text=xaxis_title, row=2, col=1,
        showline=True, linecolor="black", linewidth=1,
        showgrid=True, gridcolor="#f0f0f0",
        tickfont=dict(family='Palatino', size=10)
    )
    subplot_fig.update_yaxes(
        title_text=yaxis_title, row=2, col=1,
        showline=True, linecolor="black", linewidth=1,
        showgrid=True, gridcolor="#f0f0f0",
        tickfont=dict(family='Palatino', size=10)
    )

    return subplot_fig

make_3D_surface(
    f = lambda x, y: x * np.e ** (-x ** 2 - y ** 2),
    lim = 2,
    xaxis_title = 'x₁',
    yaxis_title = 'x₂',
    zaxis_title = 'f(x₁, x₂)',
    title=''
)

# show_surface_and_contour_side_by_side(
#     f = lambda x, y: x * np.e ** (-x ** 2 - y ** 2),
#     xaxis_title = 'x₁',
#     yaxis_title = 'x₂',
#     zaxis_title = 'f(x₁, x₂)',
#     width=700,
#     height=1000,
# )

To find $\nabla f(\vec x)$, we need to compute the partial derivatives of $f$ with respect to each component of $\vec x$. The "input variables" to $f$ are $x_1$ and $x_2$, so we need to compute $\frac{\partial f}{\partial x_1}$ and $\frac{\partial f}{\partial x_2}$, but if you'd like, replace $x_1$ and $x_2$ with $x$ and $y$ if it makes the algebra a little cleaner, and then replace $x$ and $y$ with $x_1$ and $x_2$ at the end.

$$f(\vec x) = x_1 e^{-x_1^2 - x_2^2}$$

$$\frac{\partial f}{\partial x_1} = \frac{\partial}{\partial x_1} \left( x_1 e^{-x_1^2 - x_2^2} \right) = \underbrace{1 \cdot e^{-x_1^2 - x_2^2} + x_1 \cdot e^{-x_1^2 - x_2^2} \cdot (-2x_1)}_{\text{product rule}} = (1 - 2x_1^2) e^{-x_1^2 - x_2^2}$$

$$\frac{\partial f}{\partial x_2} = \frac{\partial}{\partial x_2} \left( x_1 e^{-x_1^2 - x_2^2} \right) = \underbrace{x_1 \cdot e^{-x_1^2 - x_2^2} \cdot (-2x_2)}_{\text{chain rule}} = -2x_1 x_2 e^{-x_1^2 - x_2^2}$$

Putting these together, we have

$$\nabla f(\vec x) = \begin{bmatrix} (1 - 2x_1^2) e^{-x_1^2 - x_2^2} \\ -2x_1 x_2 e^{-x_1^2 - x_2^2} \end{bmatrix}$$

Remember, $\nabla f(\vec x)$ itself is a function. If we plug in a value of $\vec x$, we get a new vector back, which tells us the direction in which $f$ is increasing the quickest at that point. For instance,

$$\nabla f\left(\begin{bmatrix} -1 \\ 0\end{bmatrix}\right) = \begin{bmatrix} (1 - 2(-1)^2) e^{-(-1)^2 - 0^2} \\ -2(-1)(0) e^{-(-1)^2 - 0^2} \end{bmatrix} = \begin{bmatrix} -1/e \\ 0 \end{bmatrix}$$

means that at the point $\vec x = \begin{bmatrix} -1 \\ 0\end{bmatrix}$, the direction in which $f$ is increasing the quickest is $\begin{bmatrix} -1/e \\ 0 \end{bmatrix}$.

In [67]:
def plot_gradient_on_contour(f, dfx1, dfx2, point, lim=2, **kwargs):
    return make_3D_contour(
        f=f,
        dfx1=dfx1,
        dfx2=dfx2,
        lim=lim,
        with_gradient=True,
        grad_point=point,
        **kwargs
    )

def plot_gradient_on_surface(f, grad_vec, point, lim=2, **kwargs):
    """
    Plot 3D surface of f, mark the given point, and plot the gradient vector (arrow) at that point.
    
    Args:
        f: function of two variables (x, y)
        grad_vec: tuple/list/np.array of shape (2,) representing the gradient vector at 'point'
        point: tuple/list/np.array of shape (2,) representing the location (x0, y0)
        lim: limits for the plot axes
        **kwargs: keyword args for customization
        
    Returns:
        Plotly figure.
    """
    fig = make_3D_surface(
        f=f,
        lim=lim,
        xaxis_title=kwargs.get('xaxis_title', 'x₁'),
        yaxis_title=kwargs.get('yaxis_title', 'x₂'),
        zaxis_title=kwargs.get('zaxis_title', 'f(x₁, x₂)'),
        title=kwargs.get('surface_title', '3D Surface with Gradient Arrow')
    )
    x0, y0 = point
    z0 = f(x0, y0)

    # Draw the starting (base) point as a marker
    fig.add_scatter3d(
        x=[x0], y=[y0], z=[z0],
        mode='markers',
        marker=dict(size=8, color='gold'),
        showlegend=False
    )

    # Draw the gradient vector as an arrow out of that point
    dx, dy = grad_vec
    # Choose a length scaling factor for visualization
    arrow_len = kwargs.get('arrow_len', 0.5)
    arrow_color = kwargs.get('arrow_color', 'crimson')
    ax = dx * arrow_len
    ay = dy * arrow_len
    # Approximate change in z for visualization: move according to the tangent plane direction
    # dz = grad_vec @ [dx, dy] is not correct for display; instead set dz = (fx * dx + fy * dy) * arrow_len
    # But we want to visualize the arrow "rising" in z proportional to the gradient strength
    dz = (grad_vec[0]*dx + grad_vec[1]*dy) * arrow_len * 0.2  # scaling makes it visible

    fig.add_trace(dict(
        type='scatter3d',
        x=[x0, x0+ax],
        y=[y0, y0+ay],
        z=[z0, z0+dz],
        mode='lines+markers',
        marker=dict(size=0.1),
        line=dict(color=arrow_color, width=6),
        showlegend=False
    ))

    # Optionally, plot a marker at the tip of the arrow
    fig.add_scatter3d(
        x=[x0+ax], y=[y0+ay], z=[z0+dz],
        mode='markers',
        marker=dict(size=4, color=arrow_color),
        showlegend=False
    )
    return fig

def plot_gradient_side_by_side(f, dfx1, dfx2, point, lim=2, **kwargs):
    return show_surface_and_contour_side_by_side(
        f=f,
        dfx1=dfx1,
        dfx2=dfx2,
        lim=lim,
        with_gradient=True,
        grad_point=point,
        **kwargs
    )


plot_gradient_on_surface(
    f=lambda x, y: x * np.e ** (-x ** 2 - y ** 2),
    dfx1=lambda x, y: (1 - 2 * x ** 2) * np.e ** (-x ** 2 - y ** 2),
    dfx2=lambda x, y: -2 * x * y * np.e ** (-x ** 2 - y ** 2),
    grad_vec=(-1/np.e, 0),
    point=(-1, 0),
    xaxis_title='x₁',
    yaxis_title='x₂',
    zaxis_title='f(x₁, x₂)',
)

Another way is to look at its **contour plot**, which I think of as its birds-eye view when you look at the surface from above. Notice the correspondence between the colors in both graphs.

The circle-like traces in the contour plot are called **level curves**; they represent slices through the surface at a constant height. On the right, the circle labeled 0.1 represents the set of points where $f(x_1, x_2) = 0.1$. It's often more convenient to look at a contour plot rather than the surface plot when visualizing a function.


---

## Gradients of Functions that Involve Matrices and Vectors

More typically, the functions we'll need to take the gradient of will themselves be defined in terms of matrix and vector operations. In all of these examples, remember that we're working with vector-to-scalar functions.

### Example: Dot Product

Let $\vec a \in \mathbb{R}^n$ be some fixed vector (the equivalent of a constant in this context). Let's find the gradient of

$$f(\vec x) = \vec a \cdot \vec x$$

I find it helpful to think about $f(\vec x)$ in its expanded form,

$$f(\vec x) = \vec a \cdot \vec x = a_1 x_1 + a_2 x_2 + \cdots + a_n x_n$$

Remember, $\nabla f(\vec x)$ contains all of the partial derivatives of $f$, which we now need to compute.

- What is $\frac{\partial f}{\partial x_1}$? To me, that looks like $a_1$, since the first term is $a_1 x_1$ and none of the other terms involve $x_1$.
- Similarly, $\frac{\partial f}{\partial x_2} = a_2$.
- In general, $\frac{\partial f}{\partial x_i} = a_i$.

Putting these together, we get

$$\nabla f(\vec x) = \begin{bmatrix} a_1 \\ a_2 \\ \vdots \\ a_n \end{bmatrix} = \vec a$$

### Example: Norm and Chain Rule

Here's an extremely important example that shows up everywhere in machine learning. Find the gradients of:

1. $f(\vec x) = \lVert \vec x \rVert^2$
2. $f(\vec x) = \lVert \vec x \rVert$

:::{seealso} Solution
:class: dropdown

1. As we did in the previous example, we can expand $f(\vec x) = \lVert \vec x \rVert^2$ to get

    $$f(\vec x) = \vec x \cdot \vec x = x_1^2 + x_2^2 + \cdots + x_n^2$$

    For each $i$, $\frac{\partial f}{\partial x_i} = 2x_i$. So,

    $$\nabla f(\vec x) = \begin{bmatrix} 2x_1 \\ 2x_2 \\ \vdots \\ 2x_n \end{bmatrix} = \boxed{2\vec x}$$

    Think of this as the equivalent of the "power rule" for vectors.

2. There are two ways to find the gradient of $f(\vec x) = \lVert \vec x \rVert$: directly, or by using the chain rule. It's not immediately obvious how the chain rule should work here, so we'll start with the direct method and reason about how the chain rule may arise.

    **Direct method**: Let's start by expanding $f(\vec x) = \lVert \vec x \rVert$ like we did above.

    $$f(\vec x) = \sqrt{\vec x \cdot \vec x} = (\vec x \cdot \vec x)^{1/2} = (x_1^2 + x_2^2 + \cdots + x_n^2)^{1/2}$$

    For each $i$, the (regular, scalar-to-scalar) chain rule tells us that $$\frac{\partial f}{\partial x_i} = \frac{1}{2}(x_1^2 + x_2^2 + \cdots + x_n^2)^{-1/2} \cdot 2x_i = \frac{x_i}{\sqrt{x_1^2 + x_2^2 + \cdots + x_n^2}} = \frac{x_i}{\lVert \vec x \rVert}$$

    So,

    $$\nabla f(\vec x) = \begin{bmatrix} \frac{x_1}{\lVert \vec x \rVert} \\ \frac{x_2}{\lVert \vec x \rVert} \\ \vdots \\ \frac{x_n}{\lVert \vec x \rVert} \end{bmatrix} = \boxed{\frac{\vec x}{\lVert \vec x \rVert}}$$

    **Chain rule method**: Let me start by writing $f(\vec x)$ in terms of a composition of two functions.

    $$f(\vec x) = \lVert \vec x \rVert = \sqrt{\lVert \vec x \rVert^2} = h(g(\vec x))$$

    where $g(\vec x) = \lVert \vec x \rVert^2$ and $h(x) = \sqrt{x}$. Note that $g: \mathbb{R}^n \to \mathbb{R}$ is the vector-to-scalar function we found the gradient of above, and $h: \mathbb{R} \to \mathbb{R}$ is a scalar-to-scalar function.

    Then, generalizing the calculation we did with the first method, we have a "chain rule" for a function $h(g(\vec x))$ (where $h$ is scalar-to-scalar and $g$ is vector-to-scalar):

    $$\nabla f(\vec x) = \underbrace{\left(\frac{\text{d}h}{\text{d}x}(g (\vec x))\right)}_{h' (g(\vec x))} \nabla g(\vec x)$$

    Remember that $h(x) = \sqrt{x}$, so $\frac{\text{d}h}{\text{d}x}(x) = \frac{1}{2\sqrt{x}}$ and $\frac{\text{d}h}{\text{d}x}(g(\vec x)) = \frac{1}{2\sqrt{g(\vec x)}} = \frac{1}{2\lVert \vec x \rVert}$. This means

    $$f(\vec x) = \left(\frac{\text{d}h}{\text{d}x}(g (\vec x))\right) \nabla g(\vec x) = \left( \frac{1}{2\lVert \vec x \rVert} \right) 2 \vec x = \frac{\vec x}{\lVert \vec x \rVert}$$
    
    which is what we found earlier.

:::

### Example: Norm to an Exponent

Find the gradient of $f(\vec x) = \lVert \vec x \rVert^p$, where $p$ is some real number.

:::{seealso} Solution
:class: dropdown

We can treat this as a composition of two functions, $g(\vec x) = \lVert \vec x \rVert$ and $h(x) = x^p$.

$\frac{\text{d}h}{\text{d}x}(x) = p x^{p-1}$ and $\nabla g(\vec x) = \frac{\vec x}{\lVert \vec x \rVert}$. Putting these together yields

\begin{align*}
\nabla f(\vec x) 
  &= \left( \frac{\text{d}h}{\text{d}x}(g (\vec x)) \right) \nabla g(\vec x) \\
  &= p\, g(\vec x)^{p-1} \frac{\vec x}{\lVert \vec x \rVert} \\
  &= p\, \lVert \vec x \rVert^{p-1} \frac{\vec x}{\lVert \vec x \rVert} \\
  &= p\, \lVert \vec x \rVert^{p-2} \vec x
\end{align*}

:::

### Example: Log Sum Exp

If $\vec x \in \mathbb{R}^n$, we can define the **log sum exp** function as

$$f(\vec x) = \log \left( \sum_{i=1}^n e^{x_i} \right)$$

What is $\nabla f(\vec x)$? (The answer is called the **softmax function**, and comes up all the time in machine learning, when we want our models to output predicted **probabilities** in a classification problem.)

:::{seealso} Solution
:class: dropdown

Let's look at the partial derivatives with respect to each $x_i$.

$$\frac{\partial f}{\partial x_i} = \frac{\partial}{\partial x_i} \left( \log \left( \sum_{j=1}^n e^{x_j} \right) \right) = \left(\frac{1}{\sum_{j=1}^n e^{x_j}} \right) \frac{\partial}{\partial x_i} \left( \sum_{j=1}^n e^{x_j} \right) = \frac{e^{x_i}}{\sum_{j=1}^n e^{x_j}}$$

Then,

$$\nabla f(\vec x) = \begin{bmatrix} \frac{e^{x_1}}{\sum_{j=1}^n e^{x_j}} \\ \frac{e^{x_2}}{\sum_{j=1}^n e^{x_j}} \\ \vdots \\ \frac{e^{x_n}}{\sum_{j=1}^n e^{x_j}} \end{bmatrix}$$

There isn't really a way to simplify the expression using matrix-vector operations, so I'll leave it as-is. As mentioned above, the gradient we're looking at is called the **softmax function**. The softmax function maps $\mathbb{R}^n \to \mathbb{R}^n$, meaning its a vector-to-vector function.

Let's suppose we have the matrix $\begin{bmatrix} 3 \\ 5 \\ -1 \end{bmatrix}$. What does passing it through the softmax function yield?

$$\text{softmax}\left(\begin{bmatrix} 3 \\ 5 \\ -1 \end{bmatrix}\right) = \begin{bmatrix} \frac{e^3}{e^3 + e^5 + e^{-1}} \\ \frac{e^5}{e^3 + e^5 + e^{-1}} \\ \frac{e^{-1}}{e^3 + e^5 + e^{-1}} \end{bmatrix} \approx \begin{bmatrix} 0.119 \\ 0.879 \\ 0.0002 \end{bmatrix}$$

The output vector has the same number of elements as the input vector, but each element is between 0 and 1, and the sum of elements is 1, meaning that we can interpret the outputted vector as a probability distribution. Larger values in the output correspond to larger values in the input, and almost all of the "mass" is concentrated at the maximum element of the input vector (position 2), hence the name "soft" max.
:::

### Example: Quadratic Forms

Suppose $x \in \mathbb{R}^n$ and $A$ is an $n \times n$ matrix. The function $$f(\vec x) = \vec x^T A \vec x$$ is called a **quadratic form**, and its gradient is given by

$$\nabla f(\vec x) = (A + A^T) \vec x$$

We won't directly cover the proof of this formula here; one place to find it is [here](https://www.cs.ubc.ca/~schmidtm/Courses/Notes/linearQuadraticGradients.pdf). Instead, we'll focus our energy on understanding how it works.

1. Let $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$. Expand out $f(\vec x) = \vec x^T A \vec x$ and compute $\nabla f(\vec x)$ directly by computing partial derivatives, and verify that the result you get matches the formula above.
1. In quadratic forms, we typically assume that $A$ is symmetric, meaning that $A = A^T$. Why do you think this assumption is made (what does it help with)?
    - Hint: Let $A = \begin{bmatrix} 3 & 2 \\ 6 & 1 \end{bmatrix}$ and $B = \begin{bmatrix} 3 & 4 \\ 4 & 1 \end{bmatrix}$. Compute $\nabla (\vec x^T A \vec x)$ and $\nabla (\vec x^T B \vec x)$.
1. If $A$ is any symmetric $n \times n$ matrix, what is $\nabla f(\vec x)$?
1. Suppose $A$ is symmetric and $n \times n$, $\vec b \in \mathbb{R}^n$, and $c \in \mathbb{R}$. What is the gradient of

    $$f(\vec x) = \vec x^T A \vec x + \vec b \cdot \vec x + c$$

### Summary of Important Gradient Rules

| Function | Name | Gradient |
|----------|----------|----------|
| $f(\vec x) = \vec a \cdot \vec x$ | dot product | $\nabla f(\vec x) = \vec a$ |
| $f(\vec x) = \lVert \vec x \rVert^2$ | squared norm | $\nabla f(\vec x) = 2\vec x$ |
| $f(\vec x) = \vec x^T A \vec x$ | quadratic form | $\nabla f(\vec x) = (A + A^T) \vec x$<br>if $A$ is symmetric, $\nabla f(\vec x) = 2A \vec x$ |

---

## Optimization

In the calculus of scalar-to-scalar functions, we have a well-understood procedure for finding the extrema of a function. The general strategy is to take the derivative, set it to zero, and solve for the inputs (called **critical points**) that satisfy that condition. To be thorough, we'd perform a second derivative test to check whether each critical point is a maximum, minimum, or neither. 

In the land of vector-to-scalar functions, the equivalent is to solve for where the **gradient** is zero, which corresponds to finding where all partial derivatives are zero. There is a second derivative test for vector-to-scalar functions too; we'll see it later in the semester, as a thorough discussion of it involves introducing more machinery.

TODO put another basic example

---

## Minimizing Mean Squared Error

Remember, the goal of this section is to minimize mean squared error,

$$R_\text{sq}(\vec w) = \frac{1}{n} \lVert \vec y - X \vec w \rVert^2$$

In the general case, $X$ is an $n \times (d + 1)$ matrix, $y \in \mathbb{R}^n$, and $\vec w \in \mathbb{R}^{d+1}$.

We're now equipped with the tools to minimize $R_\text{sq}(\vec w)$ by taking its gradient and setting it to zero. Hopefully, we end up with the same conditions on $\vec w^*$ that we derived in [Chapter 2.10](../vectors-and-matrices/projection-2).

The goal is to **not** have to compute $\frac{\partial R_\text{sq}}{\partial w_0}$, $\frac{\partial R_\text{sq}}{\partial w_1}$, ..., $\frac{\partial R_\text{sq}}{\partial w_d}$; the gradient rules we've just established should make our work much simpler. Let's start by rewriting the squared norm as a dot product and eventually matrix multiplication.

\begin{align*}R_\text{sq}(\vec w) = \frac{1}{n} \lVert \vec y - X \vec w \rVert^2 &= \frac{1}{n} (\vec y - X \vec w) \cdot (\vec y - X \vec w) \\ &= \underbrace{\frac{1}{n} (\vec y - X \vec x)^T (\vec y - X \vec w)}_{\text{since } \vec u \cdot \vec v = \vec u^T \vec v} \\ &= \frac{1}{n} \left( \vec y^T - (X \vec w)^T \right) (\vec y - X \vec w) \\ &= \frac{1}{n} \left( \vec y^T \vec y - {\color{orange}\vec y^T X \vec w} - {\color{orange}(X \vec w)^T \vec y} + (X \vec w)^T X \vec w \right)\end{align*}

Let's focus on the two terms in <span style="color: orange; font-weight: bold;">orange</span>. They are both equal: they are both the dot product of $\vec y$ and $X \vec w$. Ideally, I want to express each term as a dot product of $\vec w$ with something, since I'm taking the gradient with respect to $\vec w$. Remember, the dot product is a **scalar**, and the transpose of a scalar is just that same scalar. So,

$$\vec y^T X \vec w = (\vec y^T X \vec w)^T = \vec w^T X^T \vec y = \vec w^T (X^T \vec y)$$

so, performing this substitution in for both orange terms gives us

\begin{align*}R_\text{sq}(\vec w) &= \frac{1}{n} \left( \vec y^T \vec y - {\color{orange}\vec w^T (X^T \vec y)} - {\color{orange}\vec w^T X^T \vec y} + \vec w^T (X^T X) \vec w \right) \\ &= \frac{1}{n} \left( \vec y^T \vec y - 2 \vec w^T (X^T \vec y) + \vec w^T (X^T X) \vec w \right)\end{align*}

Now, we're ready to take the gradient, which we'll do term by term.

- $\nabla \left( \vec y^T \vec y \right) = \vec 0$, since $\vec y^T \vec y$ is a constant with respect to $\vec w$
- $\nabla \left( 2 \vec w^T (X^T \vec y) \right) = 2 X^T \vec y$ using the dot product rule, since this is the dot product between $2X^T \vec y$ (a vector) and $\vec w$ (a vector)
- $\nabla \left( \vec w^T (X^T X) \vec w \right) = 2X^T X \vec w$, using the quadratic form rule, since $X^T X$ is a symmetric matrix

Plugging these terms in gives us

\begin{align*}R_\text{sq}(\vec w) &= \frac{1}{n} \left( \vec y^T \vec y - 2 \vec w^T (X^T \vec y) + \vec w^T (X^T X) \vec w \right) \\ \nabla R_\text{sq}(\vec w) &= \frac{1}{n} \left( \nabla \left(\vec y^T \vec y \right) - \nabla \left( 2 \vec w^T (X^T \vec y) \right) + \nabla \left( \vec w^T (X^T X) \vec w \right) \right) \\ &= \frac{1}{n} \left( 0 - 2 X^T \vec y + 2X^T X \vec w \right) \\ &= \boxed{\frac{2}{n} (X^T X \vec w - X^T \vec y)} \end{align*}

Finally, to find the minimizer $\vec w^*$, we set the gradient to zero and solve.

$$\frac{2}{n} (X^T X \vec w^* - X^T \vec y) = 0 \implies X^TX \vec w^* = X^T \vec y$$

Stop me if this feels familiar... these are the **normal equations** once again! TODO write a summary

---

## Linear Approximations

What else can the gradient tell us?

$$f(\vec x) \approx f(\vec x_0) + \nabla f(\vec x_0) \cdot (\vec x - \vec x_0)$$